# 🧠 Aula 14 — LSTM para Predição Dinâmica

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Dataset:** `reator_dinamico.csv` — CSTR com dinâmica FOPDT (τ=10min, θ=5min)

---

## Contexto

Na Aula 13 usamos MLP + lags. O problema: se θ=30 min, precisamos de 30 lags (maldição da dimensionalidade). A **LSTM** tem memória interna que aprende quantos passos de história importam.

## Célula LSTM (conceito)

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

$$h_t = o_t \odot \tanh(c_t)$$

- **Forget gate** (f): o que esquecer da memória
- **Input gate** (i): o que adicionar
- **Output gate** (o): o que passar para a saída
- **Cell state** (c): memória interna de longo prazo

## 3.1 — LSTM vs MLP: Setup e Sequências

Siga as células abaixo.

### Célula 1: Importar bibliotecas + carregar

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula13/reator_dinamico.csv"
df = pd.read_csv(URL, parse_dates=['timestamp'])
df.set_index('timestamp', inplace=True)
print(f"TensorFlow {tf.__version__}")
print(df.head())
print(df.describe().round(3))

### Célula 2: Normalizar e criar sequências

In [ ]:
WINDOW = 30   # janela temporal (passos)
HORIZON = 1   # 1 passo à frente

data = df[['F_alimentacao_L_min', 'CA_mol_L']].values

# Normalizar (fit no treino!)
scaler_X = StandardScaler()
data_n = scaler_X.fit_transform(data)

def make_sequences(data, window, horizon):
    Xs, ys = [], []
    for i in range(len(data) - window - horizon + 1):
        Xs.append(data[i:i+window])
        ys.append(data[i+window+horizon-1, 1])  # target: CA
    return np.array(Xs), np.array(ys)

X_seq, y_seq = make_sequences(data_n, WINDOW, HORIZON)
print(f"Shape sequências: X={X_seq.shape} (amostras, timesteps, features)  y={y_seq.shape}")
print("Fundamental: dados 3D para LSTM!")

### Célula 3: Split TEMPORAL (não aleatório!)

In [ ]:
# Séries temporais: split por POSIÇÃO, não aleatório
n = len(X_seq)
tr = int(n * 0.7)
va = int(n * 0.85)
X_tr, X_va, X_te = X_seq[:tr], X_seq[tr:va], X_seq[va:]
y_tr, y_va, y_te = y_seq[:tr], y_seq[tr:va], y_seq[va:]
print(f"Treino: {len(X_tr)} | Validação: {len(X_va)} | Teste: {len(X_te)}")
print("IMPORTANTE: validação temporal evita data leakage!")

### Célula 4: Definir LSTM

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(WINDOW, X_seq.shape[2])),
    layers.LSTM(64),
    layers.Dropout(0.2),
    layers.Dense(1)
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='mse', metrics=['mae'])
model.summary()

### Célula 5: Treinar com EarlyStopping

In [ ]:
callbacks = [keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
history = model.fit(X_tr, y_tr, epochs=100, validation_data=(X_va, y_va),
                    callbacks=callbacks, batch_size=32, verbose=0)

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.xlabel('Época'); plt.ylabel('Loss (MSE)')
plt.legend(); plt.grid(alpha=0.3); plt.title('Loss Curve — LSTM')
plt.show()
print(f"Épocas treinadas: {len(history.history['loss'])}")

### Célula 6: Avaliar LSTM no teste (1 passo)

In [ ]:
# O scaler normaliza X (features) e y (target) juntos.
# Para reverter a normalização do target, precisamos do scaler das colunas.
y_pred = model.predict(X_te, verbose=0)

# Reconstruir as colunas normalizadas completas (feature + target) para reverter
target_col = 1  # índice de CA_mol_L em data_n

def inverse_target(y_std, scaler_x, target_col):
    # Reinsere o valor normalizado no vetor de features (bzero no resto)
    dummy = np.zeros((len(y_std), scaler_x.n_features_in_))
    dummy[:, target_col] = y_std.ravel()
    return scaler_x.inverse_transform(dummy)[:, target_col]

y_pred_real = inverse_target(y_pred, scaler_X, target_col)
y_test_real = inverse_target(y_te, scaler_X, target_col)
rmse_lstm = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
r2_lstm = r2_score(y_test_real, y_pred_real)
print(f"LSTM (1 passo): RMSE={rmse_lstm:.4f}  R²={r2_lstm:.4f}")

### Célula 7: Baseline MLP com 30 lags

In [ ]:
from sklearn.neural_network import MLPRegressor

# MLP + 30 lags da vazão
dfm = df.copy()
feat_lags = []
for lag in range(1, 31):
    dfm[f'F_lag{lag}'] = dfm['F_alimentacao_L_min'].shift(lag)
    feat_lags.append(f'F_lag{lag}')
dfm = dfm.dropna()

X_mlp = dfm[feat_lags].values
y_mlp = dfm['CA_mol_L'].values
sc_mlp = StandardScaler()
X_mlp_n = sc_mlp.fit_transform(X_mlp)

split = int(len(X_mlp_n) * 0.8)
mlp = MLPRegressor(hidden_layer_sizes=(64,), max_iter=500, random_state=42)
mlp.fit(X_mlp_n[:split], y_mlp[:split])
y_pred_mlp = mlp.predict(X_mlp_n[split:])
rmse_mlp = np.sqrt(mean_squared_error(y_mlp[split:], y_pred_mlp))
r2_mlp = r2_score(y_mlp[split:], y_pred_mlp)
print(f"MLP + 30 lags (1 passo): RMSE={rmse_mlp:.4f}  R²={r2_mlp:.4f}")

### Célula 8: Comparação e previsão multi-passos (recursiva)

In [ ]:
print("\n=== Comparação (1 passo) ===")
print(f"MLP + 30 lags: RMSE={rmse_mlp:.4f}  R²={r2_mlp:.4f}")
print(f"LSTM (64):      RMSE={rmse_lstm:.4f}  R²={r2_lstm:.4f}")

print("\n=== Previsão multi-passos (recursiva) ===")
def forecast_recursive(model, X_seed, steps, scaler_x, target_col=1):
    X_curr = X_seed.copy()
    preds = []
    for _ in range(steps):
        p = model.predict(X_curr, verbose=0)
        preds.append(p[0, 0])
        new_row = X_curr[0, -1, :].copy()
        new_row[target_col] = p[0, 0]  # realimenta a predição
        X_curr = np.concatenate([X_curr[:, 1:, :], new_row.reshape(1, 1, -1)], axis=1)
    return np.array(preds)

seed = X_te[0:1]
steps = 10
pred_rec = forecast_recursive(model, seed, steps, scaler_X)
pred_rec_real = inverse_target(pred_rec, scaler_X, target_col)
real_rec = inverse_target(y_te[:steps], scaler_X, target_col)
print(f"LSTM multi-passo ({steps} passos): RMSE={np.sqrt(mean_squared_error(real_rec, pred_rec_real)):.4f}")

### Célula 9: Plotar predição (teste)

In [ ]:
plt.figure(figsize=(12, 5))
idx = np.arange(min(len(y_test_real), len(y_pred_mlp)))
plt.plot(idx, y_test_real[:len(idx)], label='CA real')
plt.plot(idx, y_pred_real[:len(idx)], '--', label=f'LSTM (RMSE={rmse_lstm:.4f})')
plt.plot(idx, y_pred_mlp[:len(idx)], '--', alpha=0.7, label=f'MLP+lags (RMSE={rmse_mlp:.4f})')
plt.legend(); plt.title('Predição no teste — LSTM vs MLP')
plt.xlabel('Passo'); plt.ylabel('CA (mol/L)'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Célula 10: Conclusão

> **Conclusão (3 parágrafos):**
> 1. A LSTM superou a MLP? (RMSE 1 passo + multi-passo)
> 2. A previsão multi-passos degrada com que rapidez?
> 3. O modelo é útil para controle preditivo? A LSTM é sempre superior?

---

## Checklist

- [ ] Dados convertidos em sequências (3D)
- [ ] Janela temporal definida (30)
- [ ] Split TEMPORAL (não aleatório)
- [ ] LSTM treinada com EarlyStopping
- [ ] Loss curve plotada
- [ ] MLP + lags treinada (baseline)
- [ ] Comparação 1 passo e 10 passos
- [ ] Previsão multi-passos testada
- [ ] Conclusão